In [5]:
import numpy as np
import trimesh
import time
import plotly.graph_objects as go

In [6]:
# ------------------- 1. 生成两个简单曲面 -------------------
def generate_surface1(resolution=150):
    """下表面：轻微倾斜的近似平面 + 小噪声"""
    np.random.seed(42)
    x = np.linspace(-5, 5, resolution)
    y = np.linspace(-5, 5, resolution)
    X, Y = np.meshgrid(x, y)
    Z = 0.2 * X + 0.1 * Y + 0.05 * np.random.randn(*X.shape)
    return X, Y, Z

def generate_surface2(resolution=150):
    """上表面：波浪面 + 小噪声"""
    x = np.linspace(-5, 5, resolution)
    y = np.linspace(-5, 5, resolution)
    X, Y = np.meshgrid(x, y)
    Z = 1.0 + 0.8 * np.sin(0.5 * X) * np.cos(0.5 * Y) + 0.05 * np.random.randn(*X.shape)
    return X, Y, Z

print("正在生成两个曲面...")
X1, Y1, Z1 = generate_surface1()
X2, Y2, Z2 = generate_surface2()

# ------------------- 2. 将曲面转为 Trimesh 网格 -------------------
def grid_to_trimesh(X, Y, Z):
    vertices = np.column_stack((X.ravel(), Y.ravel(), Z.ravel()))
    rows, cols = X.shape
    faces = []
    for i in range(rows - 1):
        for j in range(cols - 1):
            v1 = i * cols + j
            v2 = i * cols + j + 1
            v3 = (i + 1) * cols + j
            v4 = (i + 1) * cols + j + 1
            faces.append([v1, v2, v3])
            faces.append([v2, v4, v3])
    return trimesh.Trimesh(vertices=vertices, faces=np.array(faces))

print("正在转换为网格...")
mesh_lower = grid_to_trimesh(X1, Y1, Z1)   # 下表面
mesh_upper = grid_to_trimesh(X2, Y2, Z2)   # 上表面

# 确保法线方向一致（向上为正）
mesh_lower.fix_normals()
mesh_upper.fix_normals()


正在生成两个曲面...
正在转换为网格...


<trimesh.Trimesh(vertices.shape=(22500, 3), faces.shape=(44402, 3))>

In [7]:
def check_normals(mesh, name="网格"):
    mesh.fix_normals()
    normals = mesh.face_normals
    z_comp = normals[:, 2]
    
    print(f"\n=== {name} 法线检查 ===")
    print(f"面数: {len(normals)}")
    print(f"Z分量范围: {z_comp.min():.4f} ~ {z_comp.max():.4f}")
    print(f"Z分量平均: {z_comp.mean():.4f}")
    if np.all(z_comp > 0):
        print("✓ 所有法线朝上（Z > 0），方向一致且正确！")
    elif np.all(z_comp < 0):
        print("⚠ 所有法线朝下（Z < 0），方向一致但整体反了")
        mesh.invert_faces()
        print("已自动翻转为朝上")
    else:
        print("✗ 法线方向不一致！请检查网格生成过程")


check_normals(mesh_lower, "下表面")
check_normals(mesh_upper, "上表面")


=== 下表面 法线检查 ===
面数: 44402
Z分量范围: 0.1617 ~ 1.0000
Z分量平均: 0.6451
✓ 所有法线朝上（Z > 0），方向一致且正确！

=== 上表面 法线检查 ===
面数: 44402
Z分量范围: 0.1680 ~ 1.0000
Z分量平均: 0.6434
✓ 所有法线朝上（Z > 0），方向一致且正确！


In [ ]:
dx = X1[0, 1] - X1[0, 0]
dy = Y1[1, 0] - Y1[0, 0]

Z_diff = Z2 - Z1

# 结果 > 0 说明隆起占主导，结果 < 0 说明塌陷占主导
net_volume = np.trapezoid(np.trapezoid(Z_diff, dx=dx), dx=dy)

print(f"净体积变化: {net_volume:.6f}")

# 4. 如果你想分别看“隆起”和“塌陷”各自的量：
v_swelling = np.trapezoid(np.trapezoid(np.maximum(Z_diff, 0), dx=dx), dx=dy)
v_subsidence = np.trapezoid(np.trapezoid(np.minimum(Z_diff, 0), dx=dx), dx=dy)

print(f"总隆起体积: {v_swelling:.6f}")
print(f"总塌陷体积: {v_subsidence:.6f}")

净体积变化: 99.975315
总隆起体积: 102.544726
总塌陷体积: -2.569411


In [ ]:
# ------------------- 3. 蒙特卡洛采样 + 体积计算 -------------------
num_samples = 2_00_00  # 100万点，精度高且速度可接受
np.random.seed(42)

# 计算整体包围盒（Z方向稍扩展，避免边界误差）
bounds = np.array([
    np.minimum(mesh_lower.bounds[0], mesh_upper.bounds[0]),
    np.maximum(mesh_lower.bounds[1], mesh_upper.bounds[1])
])
box_min, box_max = bounds
box_min[2] -= 0.5
box_max[2] += 0.5
box_size = box_max - box_min
V_box = np.prod(box_size)

# 随机采样点
random_points = np.random.uniform(box_min, box_max, size=(num_samples, 3))
start_time = time.time()
print(f"正在计算 {num_samples:,} 个采样点的符号距离（SDF），请稍等10-30秒...")

# 计算符号距离
SDF_lower = trimesh.proximity.signed_distance(mesh_lower, random_points)  # 计算 random_points 中每一个点到下表面的符号距离。返回值是一个长度为 num_samples 的一维数组
SDF_upper = trimesh.proximity.signed_distance(mesh_upper, random_points)  # <0：在上表面下方
print(f"计算完成，耗时 {time.time() - start_time:.2f} 秒")

# 判断哪些点位于两个曲面之间（即体积内部）
inside_volume = (SDF_lower > 0) & (SDF_upper < 0)
inside_points = random_points[inside_volume]

fraction = np.sum(inside_volume) / num_samples
volume_diff_approx = fraction * V_box

print("\n=== 蒙特卡洛体积估算结果 ===")
print(f"包围盒体积          : {V_box:.4f}")
print(f"两个曲面之间的体积估计 : {volume_diff_approx:.4f} 立方单位")
print(f"内部点比例          : {fraction:.6f}")
print(f"估算误差（约）       : ±{np.sqrt(fraction * (1 - fraction) / num_samples) * V_box:.4f}")

# ------------------- 4. Plotly 可视化 + 保存为 HTML -------------------
print("\n正在生成交互式3D可视化...")

fig = go.Figure()

# 下表面（蓝色半透明）
fig.add_trace(go.Surface(
    x=X1, y=Y1, z=Z1,
    colorscale='Blues', opacity=0.8,
    name='下表面（近似平面）', showscale=False
))

# 上表面（橙色半透明）
fig.add_trace(go.Surface(
    x=X2, y=Y2, z=Z2,
    colorscale='Oranges', opacity=0.8,
    name='上表面（波浪面）', showscale=False
))

# 体积内部采样点（红色点云，最多显示5000个，避免浏览器卡顿）
display_num = min(50000, len(inside_points))
if display_num > 0:
    display_indices = np.random.choice(len(inside_points), display_num, replace=False)
    pts = inside_points[display_indices]
    fig.add_trace(go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=1, color='red', opacity=0.5),
        name=f'体积内部点云（约 {volume_diff_approx:.1f} 立方单位）'
    ))

fig.update_layout(
    title="两个曲面重叠 + 体积差异估计可视化<br>"
          "蓝色=下表面 橙色=上表面 红色点云=两个曲面之间的空间体积",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z (高度)',
        aspectmode='data',
        camera=dict(eye=dict(x=1.6, y=1.6, z=1.3))
    ),
    width=1100,
    height=900,
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

# 保存为独立HTML文件（可直接用浏览器打开，完整交互）
html_file = "两个曲面重叠_体积可视化.html"
fig.write_html(html_file, include_plotlyjs='cdn')

print(f"\n完成！请用浏览器打开以下文件：")
print(f"   {html_file}")
print("   → 拖拽旋转、缩放查看，红色点云清晰显示两个曲面夹出的体积空间")